# Non-stationary Example: Time-Varying Hyperparameters vs Baseline Matern

This notebook shows a case where the signal amplitude and local smoothness change over time.

We compare:
- Stationary Matern 5/2 baseline
- Time-varying outputscale (amplitude modulation)
- Time-varying lengthscale (smoothness modulation)

In [ ]:
from __future__ import annotations

import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

repo_root = pathlib.Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from gparchitect.builders.builder import build_model_from_dsl
from gparchitect.dsl.schema import (
    FeatureGroupSpec,
    GPSpec,
    KernelSpec,
    KernelType,
    ModelClass,
    TimeVaryingSpec,
    TimeVaryingTarget,
)
from gparchitect.fitting.fitter import fit_and_validate

torch.set_default_dtype(torch.double)
torch.manual_seed(21)
np.random.seed(21)

In [ ]:
# Synthetic dataset with time-varying amplitude and mild smoothness drift
n = 260
t = np.linspace(0.0, 1.0, n)
amplitude = 0.25 + 1.35 * t
phase_warp = t ** 1.35
signal = amplitude * np.sin(7.5 * math.pi * phase_warp)
noise = np.random.normal(0.0, 0.07, size=n)
y = signal + noise

df = pd.DataFrame({"time": t, "target": y, "amplitude_true": amplitude})
df.head()

In [ ]:
# EDA: check if variance increases over time
window = 25
rolling_std = df["target"].rolling(window=window, center=True).std()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].plot(df["time"], df["target"], ".", alpha=0.55, label="observed")
axes[0].set_ylabel("target")
axes[0].set_title("Observed Signal")
axes[0].legend()

axes[1].plot(df["time"], rolling_std, color="tab:green", label="rolling std")
axes[1].plot(df["time"], 0.20 * df["amplitude_true"], color="tab:red", alpha=0.8, label="scaled true amplitude")
axes[1].set_xlabel("time")
axes[1].set_ylabel("local scale")
axes[1].set_title(f"Local Variability (window={window})")
axes[1].legend()
plt.tight_layout()

print("Why time-varying hyperparameters may help: variability is not constant through time.")

In [ ]:
split_idx = int(0.7 * n)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

train_X = torch.tensor(train_df[["time"]].values, dtype=torch.double)
train_Y = torch.tensor(train_df[["target"]].values, dtype=torch.double)
test_X = torch.tensor(test_df[["time"]].values, dtype=torch.double)
test_Y = torch.tensor(test_df[["target"]].values, dtype=torch.double)

In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def nll_gaussian(y_true: np.ndarray, mean: np.ndarray, var: np.ndarray, eps: float = 1e-8) -> float:
    var_safe = np.maximum(var, eps)
    return float(np.mean(0.5 * np.log(2.0 * np.pi * var_safe) + 0.5 * ((y_true - mean) ** 2) / var_safe))


def fit_predict(spec: GPSpec, train_X: torch.Tensor, train_Y: torch.Tensor, test_X: torch.Tensor) -> dict:
    model = build_model_from_dsl(spec, train_X, train_Y)
    fit_result = fit_and_validate(model, train_X, train_Y)
    if not fit_result.success:
        raise RuntimeError(f"Fit failed: {fit_result.error_message}")

    fitted = fit_result.model
    fitted.eval()
    fitted.likelihood.eval()

    with torch.no_grad():
        posterior = fitted.posterior(test_X)
        mean = posterior.mean.squeeze(-1).cpu().numpy()
        var = posterior.variance.squeeze(-1).cpu().numpy()

    return {"model": fitted, "mean": mean, "var": var}

In [ ]:
baseline_spec = GPSpec(
    model_class=ModelClass.SINGLE_TASK_GP,
    feature_groups=[
        FeatureGroupSpec(
            name="time",
            feature_indices=[0],
            kernel=KernelSpec(kernel_type=KernelType.MATERN_52),
        )
    ],
    input_dim=1,
    output_dim=1,
)

tv_outputscale_spec = GPSpec(
    model_class=ModelClass.SINGLE_TASK_GP,
    feature_groups=[
        FeatureGroupSpec(
            name="time",
            feature_indices=[0],
            kernel=KernelSpec(
                kernel_type=KernelType.MATERN_52,
                time_varying=TimeVaryingSpec(
                    target=TimeVaryingTarget.OUTPUTSCALE,
                    time_feature_index=0,
                ),
            ),
        )
    ],
    input_dim=1,
    output_dim=1,
)

tv_lengthscale_spec = GPSpec(
    model_class=ModelClass.SINGLE_TASK_GP,
    feature_groups=[
        FeatureGroupSpec(
            name="time",
            feature_indices=[0],
            kernel=KernelSpec(
                kernel_type=KernelType.MATERN_52,
                time_varying=TimeVaryingSpec(
                    target=TimeVaryingTarget.LENGTHSCALE,
                    time_feature_index=0,
                ),
            ),
        )
    ],
    input_dim=1,
    output_dim=1,
)

In [ ]:
baseline = fit_predict(baseline_spec, train_X, train_Y, test_X)
tv_out = fit_predict(tv_outputscale_spec, train_X, train_Y, test_X)
tv_len = fit_predict(tv_lengthscale_spec, train_X, train_Y, test_X)

y_true = test_Y.squeeze(-1).cpu().numpy()

metrics = pd.DataFrame(
    [
        {
            "model": "Matern52 baseline",
            "RMSE": rmse(y_true, baseline["mean"]),
            "MAE": mae(y_true, baseline["mean"]),
            "NLL": nll_gaussian(y_true, baseline["mean"], baseline["var"]),
        },
        {
            "model": "Time-varying outputscale",
            "RMSE": rmse(y_true, tv_out["mean"]),
            "MAE": mae(y_true, tv_out["mean"]),
            "NLL": nll_gaussian(y_true, tv_out["mean"], tv_out["var"]),
        },
        {
            "model": "Time-varying lengthscale",
            "RMSE": rmse(y_true, tv_len["mean"]),
            "MAE": mae(y_true, tv_len["mean"]),
            "NLL": nll_gaussian(y_true, tv_len["mean"], tv_len["var"]),
        },
    ]
).sort_values("RMSE")

metrics

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["time"], df["target"], ".", alpha=0.2, label="observed (all)")
ax.plot(test_df["time"], y_true, color="black", linewidth=2, label="test truth")

for name, result, color in [
    ("Matern52 baseline", baseline, "tab:orange"),
    ("TV outputscale", tv_out, "tab:blue"),
    ("TV lengthscale", tv_len, "tab:green"),
]:
    mean = result["mean"]
    std = np.sqrt(np.maximum(result["var"], 1e-9))
    x = test_df["time"].to_numpy()
    ax.plot(x, mean, color=color, linewidth=2, label=f"{name} mean")
    ax.fill_between(x, mean - 1.96 * std, mean + 1.96 * std, color=color, alpha=0.14)

ax.set_title("Held-Out Predictive Comparison")
ax.set_xlabel("time")
ax.set_ylabel("target")
ax.legend(loc="best")
plt.tight_layout()

In [ ]:
# Inspect learned time-varying modulation parameters
def get_tv_params(model: object) -> tuple[float, float]:
    for name, param in model.named_parameters():
        if name.endswith("raw_tv_bias"):
            bias = float(param.detach().cpu().squeeze())
        if name.endswith("raw_tv_slope"):
            slope = float(param.detach().cpu().squeeze())
    return bias, slope

b_out, s_out = get_tv_params(tv_out["model"])
b_len, s_len = get_tv_params(tv_len["model"])

print("TV outputscale raw bias/slope:", b_out, s_out)
print("TV lengthscale raw bias/slope:", b_len, s_len)